In [1]:
import torch
import faiss
from pypdf import PdfReader
import numpy as np
import pickle

from transformers import (DPRContextEncoder, DPRContextEncoderTokenizer,
                          DPRQuestionEncoder, DPRQuestionEncoderTokenizer)

In [2]:
import os

In [3]:
device = 'mps'

In [4]:
KNOWLEDGE_BASE = "../Knowledge Bases/Coal_Mines_Regulation_2017_Noti.pdf"

In [5]:
reader = PdfReader(KNOWLEDGE_BASE)
print("Number of pages: ", len(reader.pages))

Number of pages:  280


In [6]:
def read_and_split_pdf(filename):
    reader = PdfReader(filename)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + '\n'

    paragraphs = text.split("\n")

    return [
        para.strip()
        for para in paragraphs
        if len(para.strip()) > 0
    ]

In [7]:
paragraphs = read_and_split_pdf(KNOWLEDGE_BASE)
print("Number of paragraphs: ", len(paragraphs))

Number of paragraphs:  11726


In [8]:
context_tokenizer = DPRContextEncoderTokenizer.from_pretrained('facebook/dpr-ctx_encoder-single-nq-base')
context_encoder = DPRContextEncoder.from_pretrained('facebook/dpr-ctx_encoder-single-nq-base')
context_encoder = context_encoder.to(device)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRContextEncoderTokenizer'.
Some weights of the model checkpoint at facebook/dpr-ctx_encoder-single-nq-base were not used when initializing DPRContextEncoder: ['ctx_encoder.bert_model.pooler.dense.bias', 'ctx_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRContextEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRContextEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification mod

In [9]:
def encode_contexts(text_list, batch_size=8):
    embeddings = []
    context_encoder.eval()

    for i in range(0, len(text_list), batch_size):

        batch = text_list[i:i + batch_size]

        inputs = context_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        with torch.no_grad():
            outputs = context_encoder(
                **inputs
            )

        embeddings.append(
            outputs.pooler_output.cpu()
        )

    return torch.cat(embeddings).numpy()

In [10]:
context_embeddings = encode_contexts(paragraphs)

In [13]:
MIRA_RAG_PATH = "../Knowledge Bases/Encodings/"

In [14]:
import numpy as np
import os

EMBEDDINGS_PATH = os.path.join(
    MIRA_RAG_PATH,
    "regulatory_embeddings.npy"
)

np.save(
    EMBEDDINGS_PATH,
    context_embeddings
)

print("Embeddings saved:", EMBEDDINGS_PATH)

Embeddings saved: ../Knowledge Bases/Encodings/regulatory_embeddings.npy


In [18]:
with open(
    os.path.join(
        MIRA_RAG_PATH,
        "regulatory_chunks.pkl"
    ),
    "wb"
) as f:
    pickle.dump(paragraphs, f)

---

In [2]:
import faiss
import numpy as np

print("FAISS version:", faiss.__version__)

FAISS version: 1.7.3


In [3]:
context_embeddings = np.load(
    "../Knowledge Bases/Encodings/regulatory_embeddings.npy"
)

print("Shape:", context_embeddings.shape)
print("Dtype:", context_embeddings.dtype)
print("Contiguous:", context_embeddings.flags["C_CONTIGUOUS"])

Shape: (11726, 768)
Dtype: float32
Contiguous: True


In [4]:
test_embeddings = np.ascontiguousarray(
    context_embeddings[:100],
    dtype=np.float32
)

test_index = faiss.IndexFlatL2(768)

test_index.add(test_embeddings)

test_query = test_embeddings[:1]

print("Starting FAISS 1.7.3 L2 search...")

D, I = test_index.search(
    test_query,
    5
)

print("SUCCESS")
print("Distances:", D)
print("Indices:", I)

Starting FAISS 1.7.3 L2 search...
SUCCESS
Distances: [[  0.       106.389084 107.528076 108.217285 110.07976 ]]
Indices: [[ 0 81 52 90  8]]


In [5]:
full_embeddings = np.ascontiguousarray(
    context_embeddings,
    dtype=np.float32
)

full_index = faiss.IndexFlatL2(768)

full_index.add(full_embeddings)

print("Vectors:", full_index.ntotal)

D, I = full_index.search(
    full_embeddings[:1],
    5
)

print("FULL L2 SEARCH SUCCESS")
print(D)
print(I)

Vectors: 11726
FULL L2 SEARCH SUCCESS
[[ 0.      93.96651 95.50844 97.02782 97.02782]]
[[   0 1917 5425  630  687]]


In [6]:
INDEX_PATH = "../Knowledge Bases/Encodings/regulatory.index"

faiss.write_index(
    full_index,
    INDEX_PATH
)

print("Index saved successfully")

Index saved successfully


In [7]:
loaded_index = faiss.read_index(
    INDEX_PATH
)

print("Vectors:", loaded_index.ntotal)
print("Dimension:", loaded_index.d)

D, I = loaded_index.search(
    full_embeddings[:1],
    5
)

print("RELOADED SEARCH SUCCESS")
print("Distances:", D)
print("Indices:", I)

Vectors: 11726
Dimension: 768
RELOADED SEARCH SUCCESS
Distances: [[ 0.      93.96651 95.50844 97.02782 97.02782]]
Indices: [[   0 1917 5425  630  687]]
